# Advanced Bird Species Recognition System
## EfficientNetB3 Training Notebook (CUB-200-2011)

**Instructions:**
1. Go to `Runtime` -> `Change runtime type` and select **T4 GPU**.
2. Upload your `CUB_200_2011_cropped.zip` file to the Colab files section (left menu).
3. Run the cells below.

In [ ]:
# 1. Unzip the uploaded dataset
!unzip -q CUB_200_2011_cropped.zip -d dataset
print("Dataset successfully unzipped!")

In [ ]:
# 2. Import required libraries
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
from tqdm import tqdm
from google.colab import files

In [ ]:
# 3. Run the complete training script
def train_model():
    # Configuration
    data_dir = 'dataset/CUB_200_2011_cropped' # Make sure this matches the unzipped folder structure
    
    # Check if the path exists, if it's nested (dataset/dataset/CUB...)
    if not os.path.exists(data_dir):
      data_dir = 'dataset'
        
    num_classes = 200
    batch_size = 32 # Can be increased to 64 on T4 GPU for faster training
    num_epochs = 30
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    
    print(f"Using device: {device}")
    if str(device) == "cpu":
      print("WARNING: YOU ARE USING CPU. Go to Runtime -> Change runtime type -> Select T4 GPU")
      return
    
    # 1. Data Augmentation and Normalization
    data_transforms = {
        'train': transforms.Compose([
            transforms.RandomResizedCrop(224),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(20),
            transforms.ColorJitter(brightness=0.2, contrast=0.2),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ]),
        'test': transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ]),
    }

    print("Loading data...")
    image_datasets = {
        x: datasets.ImageFolder(os.path.join(data_dir, x), data_transforms[x])
        for x in ['train', 'test']
    }
    
    dataloaders = {
        x: DataLoader(image_datasets[x], batch_size=batch_size, shuffle=True, num_workers=2)
        for x in ['train', 'test']
    }
    
    dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'test']}
    class_names = image_datasets['train'].classes
    
    print(f"Train size: {dataset_sizes['train']} | Test size: {dataset_sizes['test']}")

    # 2. Model Setup (EfficientNetB3)
    print("Initializing EfficientNetB3...")
    model = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.IMAGENET1K_V1)
    
    # Freeze base layers for initial training focusing on head
    for param in model.parameters():
        param.requires_grad = False
        
    # Replace classification head
    num_ftrs = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.5, inplace=True),
        nn.Linear(num_ftrs, 512),
        nn.ReLU(),
        nn.Dropout(p=0.5),
        nn.Linear(512, num_classes)
    )
    
    model = model.to(device)

    # Loss and Optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.classifier.parameters(), lr=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.1, patience=3)

    # 3. Training Loop
    best_acc = 0.0
    
    for epoch in range(num_epochs):
        print(f'\nEpoch {epoch+1}/{num_epochs}')
        print('-' * 10)
        
        # Unfreeze after 5 epochs to fine-tune the whole network
        if epoch == 5:
            print("Unfreezing all layers for fine-tuning...")
            for param in model.parameters():
                param.requires_grad = True
            optimizer = optim.AdamW(model.parameters(), lr=1e-5)
            
        for phase in ['train', 'test']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            pbar = tqdm(dataloaders[phase], desc=f"{phase.capitalize()}")
            for inputs, labels in pbar:
                inputs = inputs.to(device)
                labels = labels.to(device)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)
                
                pbar.set_postfix({'loss': loss.item()})

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]

            print(f'{phase.capitalize()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            if phase == 'test':
                scheduler.step(epoch_acc)
                if epoch_acc > best_acc:
                    best_acc = epoch_acc
                    torch.save(model.state_dict(), 'efficientnet_b3_best.pth')
                    print(f"Model saved with accuracy: {best_acc:.4f}")

    print(f'\nTraining complete. Best Test Accuracy: {best_acc:4f}')
    print("Triggering notebook download for weights...")
    files.download('efficientnet_b3_best.pth')

# Start training
train_model()